# EDA: E-Commerce Churn Analysis

Verified exploratory data analysis across three heterogeneous data sources.

**Sources:** `data/customers.csv`, `data/transactions.db`, `data/transactions_monthly_agg.parquet`

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Ensure cwd is project root so data/ paths resolve correctly
if not os.path.isdir("data"):
    os.chdir(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

from eda.load import load_customers, load_transactions_db, load_transactions_agg
from eda.profile import schema_summary, distribution_stats, outlier_count_iqr, count_zero_activity_customers, churn_rate_by_group
from eda.hypotheses import run_all_hypothesis_tests
from eda.time_checks import monthly_tx_volume, check_multi_source_agreement, monthly_churn_rate_by_signup_cohort
from eda.leakage_checks import check_days_since_last_txn_separation, check_as_of_date_discipline
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
df_cust, _ = load_customers()
df_tx, _ = load_transactions_db()
df_pq, _ = load_transactions_agg()
print("All three sources loaded.")

## 1. Schema & Missingness

In [ ]:
for name, df in [("customers.csv", df_cust), ("transactions.db", df_tx), ("transactions_monthly_agg.parquet", df_pq)]:
    summary = schema_summary(df, name)
    print(f"=== {name} ({summary['rows']:,} rows) ===")
    for c in summary["columns"]:
        mid = f"missing={c['n_missing']} ({c['pct_missing']}%)" if c['n_missing'] > 0 else "no missing"
        uid = f" | unique={c['n_unique']}" if c['n_unique'] is not None else ""
        print(f"  {c['column']:20s}  {c['dtype']:12s}  {mid}{uid}")
    print()

## 2. Churn Rate & Class Imbalance

In [ ]:
n_churned = df_cust["churned"].sum()
n_total = len(df_cust)
cr = n_churned / n_total
print(f"Churned: {n_churned} ({cr*100:.1f}%)")
print(f"Not churned: {n_total - n_churned} ({(1-cr)*100:.1f}%)")
print(f"Ratio (non-churned : churned) = {(1-cr)/cr:.1f} : 1")
print(f"A dummy classifier achieves {(1-cr)*100:.1f}% accuracy with 0% recall on churn.")
print("F1-score, precision-recall AUC, and ROC-AUC are the correct metrics.")

## 3. Zero-Activity Customers

In [ ]:
za = count_zero_activity_customers(df_cust, df_tx)
print(f"Zero-activity customers: {za['n_zero_activity']} ({za['pct_of_customers']}% of total)")
print(f"Churned among zero-activity: {za['n_churned']} ({za['pct_churned']}%)")
print(f"Share of all churned customers: {za['pct_of_all_churned']}%")
print("These are NOT missing data. They signed up but never transacted.")

## 4. Churn by Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["acquisition_channel", "region", "signup_year"]):
    dfp = df_cust.copy()
    if col == "signup_year":
        dfp[col] = pd.to_datetime(dfp["signup_date"]).dt.year
    rates = dfp.groupby(col)["churned"].mean().sort_values()
    colors = ["#d32f2f" if v > 0.14 else "#1976d2" for v in rates.values]
    rates.plot(kind="barh", ax=ax, color=colors)
    ax.set_title(f"Churn Rate by {col.replace('_', ' ').title()}")
    ax.set_xlabel("Churn Rate")
    for i, v in enumerate(rates.values):
        ax.text(v + 0.005, i, f"{v*100:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Transaction Profiling by Churn Status

In [ ]:
tx_agg = df_tx.groupby("customer_id").agg(
    total_spend=("amount", "sum"), txn_count=("amount", "count"),
    avg_txn=("amount", "mean"), max_txn=("amount", "max"),
    return_count=("is_returned", "sum"), distinct_categories=("category", "nunique"),
    distinct_channels=("channel", "nunique"),
).reset_index()
merged = df_cust.merge(tx_agg, on="customer_id", how="left")
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
metrics = [("total_spend", "Total Spend ($)"), ("txn_count", "Transaction Count"),
           ("distinct_categories", "Distinct Categories"), ("avg_txn", "Avg Txn Value ($)"),
           ("return_count", "Return Count"), ("distinct_channels", "Distinct Channels")]
for ax, (col, label) in zip(axes.flatten(), metrics):
    data = [merged[merged["churned"]==0][col].dropna(), merged[merged["churned"]==1][col].dropna()]
    bp = ax.boxplot(data, labels=["Not Churned", "Churned"], patch_artist=True)
    bp["boxes"][0].set_facecolor("#1976d2")
    bp["boxes"][1].set_facecolor("#d32f2f")
    ax.set_title(label)
plt.tight_layout()
plt.show()

In [ ]:
results = run_all_hypothesis_tests(df_cust, df_tx)
print(f"{'Driver':<30s} {'Test':<18s} {'p-value':<10s} {'Significant?':<12s} {'Effect':<10s}")
print("-"*80)
for r in results:
    sig = "YES" if r["significant"] else "no"
    print(f"{r['label']:<30s} {r['test']:<18s} {r['p_value']:<10.6f} {sig:<12s} {r['effect_size']:<10.4f}")

## 6. Category & Channel Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_tx["category"].value_counts().plot(kind="bar", ax=axes[0], color="#1976d2")
axes[0].set_title("Transactions by Category"); axes[0].set_ylabel("Count")
df_tx["channel"].value_counts().plot(kind="bar", ax=axes[1], color="#388e3c")
axes[1].set_title("Transactions by Channel"); axes[1].set_ylabel("Count")
plt.tight_layout(); plt.show()
cat_txns = df_tx.groupby(["customer_id","category"]).size().reset_index(name="n")
cat_churn = df_cust.merge(cat_txns, on="customer_id")
cat_pivot = cat_churn.groupby(["category","churned"])["n"].mean().unstack()
cat_pivot.columns = ["Not Churned","Churned"]
print("Mean transactions per category by churn status:")
print(cat_pivot.round(2))

## 7. Transaction Amount Distribution

In [ ]:
amt_stats = distribution_stats(df_tx["amount"])
amt_out = outlier_count_iqr(df_tx["amount"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_tx["amount"].hist(bins=50, ax=axes[0], color="#1976d2", edgecolor="white")
axes[0].set_title(f"Amount Distribution (mean=${amt_stats['mean']:.0f}, median=${amt_stats['median']:.0f})")
axes[0].set_xlabel("Amount ($)"); axes[0].set_ylabel("Frequency")
df_tx["amount"].hist(bins=50, ax=axes[1], color="#d32f2f", edgecolor="white", range=(0, 200))
axes[1].set_title(f"Zoomed (0-$200) — {amt_out['n_outliers_high']} outliers ({amt_out['pct_outliers']}%)")
axes[1].set_xlabel("Amount ($)"); axes[1].set_ylabel("Frequency")
plt.tight_layout(); plt.show()
print(f"Amount stats: mean=${amt_stats['mean']:.2f}, median=${amt_stats['median']:.2f}, skew={amt_stats['skew']}")
print(f"IQR outliers: {amt_out['n_outliers_total']} ({amt_out['pct_outliers']}%)")

## 8. Time Trend & Non-Stationarity

In [ ]:
volume = monthly_tx_volume(df_tx)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
volume.plot(x="txn_month_str", y="txn_count", kind="line", marker="o", ax=axes[0], color="#1976d2")
axes[0].set_title("Monthly Transaction Volume"); axes[0].set_xlabel("Month")
axes[0].tick_params(axis="x", rotation=45)
cohort = monthly_churn_rate_by_signup_cohort(df_cust)
cohort.plot(x="signup_month_str", y="churn_rate", kind="line", marker="o", ax=axes[1], color="#d32f2f")
axes[1].set_title("Churn Rate by Signup Month Cohort"); axes[1].set_xlabel("Signup Month")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
dfcc = df_cust.copy()
dfcc["signup_year"] = pd.to_datetime(dfcc["signup_date"]).dt.year
yearly = dfcc.groupby("signup_year")["churned"].agg(["count","sum"])
yearly["rate"] = (yearly["sum"]/yearly["count"]*100).round(1)
print("Churn rate by signup year:")
print(yearly.to_string())
print("Time non-stationarity detected: churn triples from 9.1% (2024) to 27.8% (2026).")
print("Mandatory: time-based train/test split (not random).")

## 9. Multi-Source Agreement (DB vs Parquet)

In [ ]:
msa = check_multi_source_agreement(df_tx, df_pq)
print(f"DB re-aggregated rows: {msa['total_rows_db']}")
print(f"Parquet rows: {msa['total_rows_pq']}")
print(f"Rows only in DB: {msa['n_left_only']}")
print(f"Rows only in parquet: {msa['n_right_only']}")
print(f"Matched rows with count mismatch: {msa['n_count_mismatch']}")
print(f"Mismatched months: {msa['mismatched_months']}")
print(f"July 2026 DB: ${msa['db_july_total']:.2f}, Parquet: ${msa['pq_july_total']:.2f}")
print(f"Shortfall: {msa['july_pct_shortfall']}%")
print("The parquet file is incomplete for July 2026. Use the raw DB as source of truth.")

## 10. Leakage — Label Circularity Check

In [ ]:
leak = check_days_since_last_txn_separation(df_cust, df_tx)
print(f"Churned min days_since_last_txn: {leak['churned_min_days_since_last_txn']}")
print(f"Non-churned max days_since_last_txn: {leak['non_churned_max_days_since_last_txn']}")
print(f"Separation gap: {leak['separation_gap']} day(s)")
print(f"Perfect separation: {leak['perfect_separation']}")
print(f"Correlation with churn: {leak['correlation_with_churn']}")
print(f"Zero-activity churned: {leak['n_zero_activity_and_churned']}")
print()
print("CRITICAL: days_since_last_txn perfectly separates churned from non-churned.")
print("This is definitionally circular with the churn label (90-day inactivity window).")
print("Exclude from all models: days_since_last_txn, trailing_90d_*, inactive_90d_flag.")

## 11. Model-Family Recommendation

In [ ]:
print("="*60)
print("RECOMMENDED: Gradient Boosted Trees (XGBoost / LightGBM)")
print("="*60)
print("""
Why GBT for THIS dataset:
1. Non-linear channel x spend interaction (low-spend: 15-22% churn,
   high-spend: 4-8%) captured natively by tree splits.
2. Class imbalance (6.4:1) via scale_pos_weight.
3. Robust to right-skewed monetary variables.
4. Mixed data types without one-hot explosion.
5. Feature importance for stakeholder interpretability.

Alternative considered: Random Forest (dilutes 2026 cohort signal).
Excluded (leakage): days_since_last_txn, trailing_90d_*, inactive_90d.
Usable: total_spend, txn_count, distinct_categories, return_rate,
        customer_tenure_days, signup_year, channel, region.""")

## 12. Summary of Key Findings

In [ ]:
print("="*60)
print("SUMMARY OF KEY FINDINGS")
print("="*60)
cr = df_cust["churned"].mean()
print(f" 1. Churn rate: {cr*100:.1f}% (6.4:1 ratio) — use F1/AUC, not accuracy")
print(f" 2. Zero-activity: {za['n_zero_activity']} customers, all churned — distinct subpopulation")
print(f" 3. Time drift: churn triples (9.1% -> 27.8%) — time-based split needed")
print(f" 4. Lower spend & engagement -> higher churn (p<0.001)")
print(f" 5. Avg txn value NOT significant (p=0.18) — drop from feature set")
print(f" 6. Channel & region not significant alone — use as interactions")
print(f" 7. days_since_last_txn: corr=0.74, perfect separation — LEAKAGE, exclude")
print(f" 8. Parquet incomplete (19.3% shortfall Jul 2026) — use DB instead")
print(f" 9. Recommended model: Gradient Boosted Trees")